# Match metadata: active runbook (local full pipeline)

This notebook is the single active path for production-like local runs.
Use full local pipeline stages and treat this as the source of truth for generated sessions.

- Prerequisite: transcription QC completed and good-only parquet available.
- Run order: similarity build/match/merge then metadata assignment.
- Optional: ALLSHEETS residual pass is in a dedicated notebook.


## Baseline similarity matching (RR monthly vs ensemble consensus, DuckDB/Parquet)

Every daily transcription (ensemble file) is matched to the Rainfall-Rescue
monthly records by comparing month-by-month values:

- RR vectors: station-year monthly profiles from monthly_rainfall
- Ensemble vectors: monthly values from all 5 ensemble members
- Primary score: count of months where RR equals any ensemble member
  (after rounding both values to 2 decimal places)
- Tie-breaker: higher overlap-month count
- Compatibility fields: cosine and adjusted score are still stored
  for diagnostics and historical comparability

The cell below runs this matcher interactively on a bounded slice so the
notebook stays fast. Use the SLURM scripts for full-scale matching.

In [1]:
# Setup Parquet roots for RR, filtered ensemble, and similarity datasets.

import os
from pathlib import Path

from src.rainfall_rescue_sqlite.ingest import default_db_path
from src.rainfall_rescue_sqlite.ensemble_ingest import default_ensemble_db_path
from src.rainfall_rescue_sqlite.parquet_ingest import default_rainfall_rescue_parquet_root
from src.rainfall_rescue_sqlite.parquet_similarity import default_comparison_parquet_root

# Metadata matching uses only sources that passed transcription QC and were
# deduplicated by the full-scale transcription-QC merge.
rr_dataset_root = default_rainfall_rescue_parquet_root()
ensemble_dataset_root = Path(os.environ["PDIR"]) / "ensemble_transcriptions_parquet_good"
comparison_root = default_comparison_parquet_root()

# Legacy SQLite paths kept for downstream legacy sections in this notebook.
db_path = default_db_path()
ensemble_db_path = default_ensemble_db_path()
comparison_db_path = Path(f"{os.getenv('PDIR')}/monthly_similarity.sqlite")

rr_dataset_root, ensemble_dataset_root, comparison_root

(PosixPath('/Volumes/Scratch/ADRQ/Rainfall-Rescue/rainfall_rescue_parquet'),
 PosixPath('/Volumes/Scratch/ADRQ/ensemble_transcriptions_parquet_good'),
 PosixPath('/Volumes/Scratch/ADRQ/monthly_similarity_parquet'))

In [2]:
# Build comparison vectors from RR + ensemble parquet datasets.
#
# MEMORY NOTE: this step loads all RR vectors (~285k station-years) and all
# ensemble consensus vectors (~514k files) into Python memory to compute
# monthly medians and IQRs before writing them to parquet. On the full dataset
# that requires ~8-16 GB RAM, so it is usually run by the local staged pipeline
# (`scripts/slurm/submit_all.sh`) rather than interactively.
#
# Set rebuild_vectors = True only if you want to (re)build from scratch in this
# notebook. If comparison_root already contains vectors from a prior pipeline
# run, leave it False and go straight to the matching cell below.

from src.rainfall_rescue_sqlite.parquet_similarity import build_comparison_vectors_parquet

rebuild_vectors = False  # set True to rebuild; leave False if vectors already exist

if rebuild_vectors:
    build_result = build_comparison_vectors_parquet(
        rr_dataset_root=rr_dataset_root,
        ensemble_dataset_root=ensemble_dataset_root,
        comparison_root=comparison_root,
    )
    print(build_result)
else:
    rr_vec_path = comparison_root / "rr_monthly_vectors"
    ens_vec_path = comparison_root / "ensemble_consensus_vectors"
    if rr_vec_path.exists() and ens_vec_path.exists():
        print(f"Skipping build - vectors already present in {comparison_root}")
    else:
        print(
            "WARNING: comparison_root has no vectors yet.\n"
            "Either set rebuild_vectors = True (needs ~8-16 GB RAM for full dataset),\n"
            "or run the local full pipeline first:\n"
            "  scripts/slurm/submit_all.sh"
        )

Skipping build - vectors already present in /Volumes/Scratch/ADRQ/monthly_similarity_parquet


In [3]:
# Run bounded similarity matching against existing comparison vectors.
#
# max_ensemble_queries and max_rr_candidates cap how many vectors are loaded,
# so this cell is safe to run on a workstation regardless of dataset size.
# The matching itself keeps only the RR candidate matrix in RAM (~27 MB for
# 285k candidates) and streams ensemble queries - it does not load all vectors.
#
# For a full-scale run use the local similarity pipeline instead
# (`scripts/slurm/submit_all.sh`), which parallelises queries
# across shards and merges them into one session.

from src.rainfall_rescue_sqlite.parquet_similarity import run_baseline_matching_parquet

match_result = run_baseline_matching_parquet(
    comparison_root=comparison_root,
    top_k=10,
    min_overlap=10,
    uncertainty_weight=0.15,
    max_ensemble_queries=200,    # remove limits for full-scale run
    max_rr_candidates=20000,
)

print(match_result)

MatchResult(comparison_root=PosixPath('/Volumes/Scratch/ADRQ/monthly_similarity_parquet'), session_id=3, ensemble_queries=200, rr_candidates=20000, matches_written=1950)


In [4]:
# Inspect top exact matches from the latest parquet similarity session.

import duckdb

conn = duckdb.connect()
try:
    latest_session = conn.execute(
        f"SELECT MAX(session_id) FROM read_parquet('{comparison_root / 'similarity_sessions' / '*.parquet'}')"
    ).fetchone()[0]

    rows = conn.execute(
        f"""
        SELECT
            m.query_rank,
            m.exact_agreement_count,
            m.adjusted_score,
            m.cosine_similarity,
            m.overlap_months,
            m.ensemble_uncertainty,
            e.file_name,
            e.descriptor,
            r.station_file_id,
            r.year,
            r.location_name
        FROM read_parquet('{comparison_root / 'similarity_matches' / '*.parquet'}') m
        JOIN read_parquet('{comparison_root / 'ensemble_consensus_vectors' / '*.parquet'}') e
          ON e.ensemble_vector_id = m.ensemble_vector_id
        JOIN read_parquet('{comparison_root / 'rr_monthly_vectors' / '*.parquet'}') r
          ON r.rr_vector_id = m.rr_vector_id
        WHERE m.session_id = ?
          AND m.query_rank = 1
          AND m.exact_agreement_count = 11
        ORDER BY m.adjusted_score DESC
        LIMIT 20
        """
        , [latest_session]
    ).fetchall()
finally:
    conn.close()

print(f"Latest session: {latest_session}")
for row in rows:
    print(
        f"rank={row[0]:>2}  exact={row[1]:>2}  "
        f"overlap={row[4]:>2}  score={row[2]:.4f}  "
        f"cos={row[3]:.4f}  "
        f"unc={row[5] if row[5] is not None else 'n/a'}  "
        f"ensemble={row[6]}  rr={row[8]}:{row[9]}  "
        f"loc={row[10] or 'n/a'}"
    )

Latest session: 3
rank= 1  exact=11  overlap=12  score=0.9395  cos=0.9921  unc=0.35041666666666677  ensemble=DRain_1871-1880_Kent_Part1-16.json  rr=ACRISE-ELHAM/ACRISE-ELHAM:1873  loc=ACRISE-ELHAM
rank= 1  exact=11  overlap=12  score=0.2570  cos=0.4393  unc=1.215833333333333  ensemble=DRain_1881-1890_Kent_Part1-63.json  rr=ASHFORD-BETHERSDEN/ASHFORD-BETHERSDEN:1883  loc=ASHFORD-BETHERSDEN


## Full-scale matching on this machine

The interactive matching above only processes a small slice. To match every
good, deduplicated ensemble source against every RR station-year, the work is
split into shards and run locally with the same staged scripts used on the
cluster. All scripts use the **DuckDB/Parquet** backend by default.

Run the full transcription-source QC pipeline first. Its merge stage publishes
`$PDIR/ensemble_transcriptions_parquet_good`, containing only sources that are
not flagged bad and one representative from each duplicate group. The matching
build reads this filtered dataset through `MATCH_ENSEMBLE_PARQUET_ROOT`.

**How the work is sharded**

- A single **build** stage reads all RR vectors (~285 k station-years) and the
  good-only ensemble consensus vectors from `MATCH_ENSEMBLE_PARQUET_ROOT`, then
  writes normalised vector parquet files to `$COMPARISON_PARQUET_ROOT`.
- The ensemble queries are divided into `NUM_SHARDS` (default 100) contiguous
  slices (by `ORDER BY ensemble_vector_id`). Each local array task matches its
  own slice against all RR candidates and writes top-K results to a private
  `similarity_shard_XXXXX.parquet` file in `$SIMILARITY_SHARD_DIR`.
- A final **merge** stage reads all shard files and writes one
  `similarity_sessions` / `similarity_matches` Parquet session to
  `$COMPARISON_PARQUET_ROOT`.

**Pipeline (three stages, run locally in order)**

| Stage | Script | Local runner | Purpose |
|-------|--------|--------------|---------|
| build | `scripts/slurm/build_vectors.sbatch` | `scripts/slurm/submit_all.sh` | build comparison vectors from good-only ensemble sources |
| match | `scripts/slurm/match_array.sbatch` | `scripts/slurm/submit_all.sh` | run each shard in parallel on this machine |
| merge | `scripts/slurm/merge_shards.sbatch` | `scripts/slurm/submit_all.sh` | consolidate shards into one session |

Shard count, local concurrency and matching parameters are configured in
`scripts/slurm/config.sh` and `scripts/slurm/config.sh` (`NUM_SHARDS`,
`SLURM_QOS`, `MATCH_CORES`, `MATCH_MEM_MB`, `TOP_K`,
`MIN_OVERLAP`, `UNCERTAINTY_WEIGHT`, `BATCH_SIZE`).

Key Parquet paths are:

| Variable | Default value |
|----------|---------------|
| `MATCH_ENSEMBLE_PARQUET_ROOT` | `$PDIR/ensemble_transcriptions_parquet_good` |
| `COMPARISON_PARQUET_ROOT` | `$PDIR/monthly_similarity_parquet` |
| `SIMILARITY_SHARD_DIR` | `$PDIR/similarity_shards_parquet` |

**Launch the whole pipeline** (from the repository root, with `PDIR` set as usual):

```bash
scripts/slurm/submit_all.sh
```

Use another good-only dataset location for one run:

```bash
MATCH_ENSEMBLE_PARQUET_ROOT=/path/to/ensemble_transcriptions_parquet_good_v2 \
scripts/slurm/submit_all.sh
```

To rerun just the matching stages from an existing vector build, execute the
array and merge scripts directly in order:

```bash
scripts/slurm/submit_all.sh --skip-build
bash scripts/slurm/merge_shards.sbatch
```

**Monitor it**

```bash
ls -t $PDIR/slurm_logs | head
```

The cell below inspects the resulting session in `$COMPARISON_PARQUET_ROOT`.

In [5]:
# Summarise similarity sessions from monthly_similarity_parquet.

conn = duckdb.connect()
try:
    sessions = conn.execute(
        f"""
        SELECT session_id, status, started_at, completed_at,
               ensemble_queries, rr_candidates, matches_written
        FROM read_parquet('{comparison_root / 'similarity_sessions' / '*.parquet'}')
        ORDER BY session_id DESC
        """
    ).fetchall()
finally:
    conn.close()


def _n(value):
    # Some sessions (e.g. still running or failed) may have NULL counts.
    return "n/a" if value is None else f"{value}"


for s in sessions:
    print(
        f"session {s[0]:>3}  {str(s[1]):<8}  "
        f"queries={_n(s[4]):>6}  candidates={_n(s[5]):>7}  "
        f"matches={_n(s[6]):>8}  "
        f"({s[2]} -> {s[3]})"
    )


session   3  success   queries=   200  candidates=  20000  matches=    1950  (2026-08-25T15:59:23+00:00 -> 2026-08-25T15:59:25+00:00)
session   2  success   queries=   200  candidates=  20000  matches=    1950  (2026-08-25T15:06:58+00:00 -> 2026-08-25T15:06:59+00:00)
session   1  success   queries= 60678  candidates= 422125  matches=  606780  (2026-08-25T15:01:08+00:00 -> 2026-08-25T15:01:09+00:00)


## Assign Rainfall Rescue metadata to ensemble records

The similarity matches from the full-scale run (or interactive session) are used to enrich each ensemble file record with station metadata from Rainfall Rescue. This enables downstream analysis to work with well-georeferenced data.

**Matching strategy**

- **Exact match** (rank-1 with exact_agreement_count ≥ 9, *and* no more than 3 monthly values exactly 0 in the ensemble transcription vector): copy all metadata
  - location_name, year, latitude, longitude, elevation_ft
  - The zero-month guard rejects spurious matches from no-data transcriptions that agree with a reference station only on empty months.
- **Approximate match** (top-3 ranks by cosine score): assign conditional metadata
  - Year: only if all 3 top ranks have the same year
  - Position: only if all 3 are within 1.0° Euclidean distance → compute centroid
  - Both conditions must pass; if either fails, all metadata stays NULL
  - Location name and elevation always NULL for approximate matches
- **Unmatched**: all metadata stay NULL

The cell below clears any prior metadata and recomputes assignments from the latest comparison session.

In [6]:
# Assign RR metadata to ensemble records (DuckDB/Parquet backend)
#
# Reads the latest similarity session from comparison_root and writes a
# per-session ensemble_metadata parquet table (one row per ensemble file;
# NULL metadata for unmatched files). Parquet is immutable, so this replaces
# the old in-place SQLite UPDATE with a fresh output table.

import importlib

import duckdb

import src.rainfall_rescue_sqlite.assign_ensemble_metadata as _assign_metadata

_assign_metadata = importlib.reload(_assign_metadata)
assign_ensemble_metadata_parquet = _assign_metadata.assign_ensemble_metadata_parquet

result = assign_ensemble_metadata_parquet(
    comparison_root=comparison_root,
    ensemble_dataset_root=ensemble_dataset_root,
    rr_dataset_root=rr_dataset_root,
    session_id=None,  # Use latest session
)

print("Metadata Assignment Summary")
print(f"  Session: {result.session_id}")
print(f"  Output: {result.output_path}")
print(f"  Total ensemble files: {result.total_ensemble_files}")
print(f"  Exact matches: {result.exact_matches} ({100*result.exact_matches/result.total_ensemble_files:.1f}%)")
print(f"  Approximate matches: {result.approximate_matches} ({100*result.approximate_matches/result.total_ensemble_files:.1f}%)")
print(f"  Unmatched: {result.unmatched} ({100*result.unmatched/result.total_ensemble_files:.1f}%)")

# Sample assigned metadata from the parquet output.
sample_conn = duckdb.connect()
try:
    print("\n=== Exact Matches Sample ===")
    exact_samples = sample_conn.execute(
        f"""
        SELECT file_name, matched_location_name, matched_year,
               matched_latitude, matched_longitude, matched_elevation_ft
        FROM read_parquet('{result.output_path}')
        WHERE match_type = 'exact'
        LIMIT 5
        """
    ).fetchall()
    for row in exact_samples:
        elev = "n/a" if row[5] is None else f"{row[5]}"
        print(
            f"  {row[0]}: {row[1]} ({row[2]}) "
            f"lat={row[3]:.2f} lon={row[4]:.2f} elev={elev}"
        )

    print("\n=== Approximate Matches Sample ===")
    approx_samples = sample_conn.execute(
        f"""
        SELECT file_name, matched_year, matched_latitude, matched_longitude
        FROM read_parquet('{result.output_path}')
        WHERE match_type = 'approximate'
        LIMIT 5
        """
    ).fetchall()
    for row in approx_samples:
        print(
            f"  {row[0]}: year={row[1]} "
            f"centroid=({row[2]:.2f}, {row[3]:.2f})"
        )
finally:
    sample_conn.close()


Metadata Assignment Summary
  Session: 3
  Output: /Volumes/Scratch/ADRQ/monthly_similarity_parquet/ensemble_metadata/session_000003.parquet
  Total ensemble files: 62216
  Exact matches: 11 (0.0%)
  Approximate matches: 0 (0.0%)
  Unmatched: 62205 (100.0%)

=== Exact Matches Sample ===
  DRain_1871-1880_Bedfordshire-17.json: APSLEY-GUISE-OAKLANDS (1874) lat=52.01 lon=-0.63 elev=433.0
  DRain_1871-1880_Bedfordshire-25.json: APSLEY-GUISE-OAKLANDS (1880) lat=52.01 lon=-0.63 elev=433.0
  DRain_1871-1880_Berkshire-23.json: ALCESTER SAMBOURNE (1872) lat=52.27 lon=-1.92 elev=344.0
  DRain_1871-1880_Brecknockshire-25.json: ABERDARE NANTHIR RESERVOIR (1878) lat=51.75 lon=-3.47 elev=860.0
  DRain_1871-1880_Carnarvonshire-3.json: ABERDARON-COCHYMOEL (1872) lat=52.85 lon=-4.62 elev=340.0

=== Approximate Matches Sample ===


### Full-scale run note

The cells above are intended for bounded, interactive checking.

For a full-scale local parallel run, use the operational workflow in
`match_metadata_operations.ipynb`.